![NumPy introductory illustration](images/numpy-intro.png)


**NumPy** is the foundational library for numerical computing in Python. It gives you fast, memory-efficient arrays and the mathematical operations needed to work with them, replacing slow Python loops with vectorized calculations that run in compiled code. This speed and convenience make NumPy the base layer for the rest of the data science stack: `Pandas` stores its data in NumPy arrays, and libraries like `SciPy`, `scikit-learn`, and `TensorFlow` build directly on NumPy's array operations. Understanding NumPy makes it easier to see what these higher-level tools are doing under the hood, and to fix problems when they don't behave as expected.


By the end of this chapter you should be able to:

- Create arrays and interpret `shape`, `ndim`, `size`, and `dtype`.
- Translate pandas selection ideas into positional NumPy indexing.
- Explain how basic slices and explicit copies behave when updated.
- Predict broadcasting results from the shapes of the operands.
- Choose an aggregation axis and interpret the resulting units and shape.
- Distinguish minimum/maximum values from their positions, including ties.
- Reshape, transpose, and concatenate arrays without losing their meaning.

Complete the [practice activity](#practice-activity-shapes-sales-and-search) after the worked examples. The activity uses small synthetic sales arrays; no external data or live service is needed.

## Set Up Your Practice Files {#set-up-the-chapter-files}

Download the [NumPy Fundamentals practice kit](downloads/numpy-fundamentals-practice.zip). Extract `stat303-numpy-fundamentals` inside the `stat303-setup` project from Chapters 1–2 and select that project's verified Python environment.

```text
stat303-setup/
├── .venv/
└── stat303-numpy-fundamentals/
    ├── numpy_examples.ipynb
    ├── activity05.ipynb
    ├── README.md
    └── data/
        └── country-capital-lat-long-population.csv
```

Run `numpy_examples.ipynb` for the lesson and complete `activity05.ipynb` for your own activity report. Use `stat303-numpy-fundamentals` as the notebook working folder. The capital data support the longer independent exercise; all other examples define their inputs in Python.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
import sys

print('NumPy version:', np.__version__)
print('Working folder:', Path.cwd().name)
print('Capital exercise file found:', (Path('data') / 'country-capital-lat-long-population.csv').is_file())


NumPy version: 2.5.3
Working folder: stat303-numpy-fundamentals
Capital exercise file found: True


In [2]:
np.random.seed(303)  # Reproducible legacy examples; new examples use default_rng.


**Environment check:** NumPy and pandas are included in the environment setup from [Chapter 2](python_venv.ipynb).

If an import fails, check the **selected notebook kernel** first. If a package is missing, run the following command in the terminal with the **project environment activated**:

```bash
python -m pip install numpy pandas
```

See [Project Environments](python_venv.ipynb) for verification steps.


## Why NumPy?

NumPy arrays let you write calculations on many numbers at once, store numerical data compactly, and run many operations using fast, compiled code instead of Python loops.

### Clearer Numerical Code

Suppose we want to double each value in a list. With a Python list, we loop through the values one at a time. With a NumPy array, we can apply the calculation to the whole array at once:


In [ ]:
values = [10, 20, 30]

# Python list: calculate one value at a time.
doubled_list = []
for value in values:
    doubled_list.append(value * 2)

# NumPy array: apply the calculation to every element.
values_array = np.array(values)
doubled_array = values_array * 2

print("Python list:", doubled_list)
print("NumPy array:", doubled_array)


Python list: [20, 40, 60]
NumPy array: [20 40 60]


Both results contain `20`, `40`, and `60`. The expression `values_array * 2` multiplies every element by 2 in one step, with no loop needed. This is called a **vectorized operation**: an operation applied to an entire array without writing an explicit Python loop.

Watch out for this common mix-up: `values * 2` on the original Python list repeats the list, producing `[10, 20, 30, 10, 20, 30]`. Converting the list to a NumPy array is what changes what `*` means.

### Compact Numerical Storage

A NumPy array stores its elements using a single data type, called its **dtype**. For numerical data, this typically uses less memory than a Python list or tuple, which store references to separate Python objects instead of the raw numbers.

The example below compares memory use for the integers 0 through 999. `array.nbytes` reports the bytes used by the array's element data.


This comparison is approximate: the NumPy figure counts only the element data, while the list and tuple figures include their container overhead and the individual integer objects (which Python may share across values). Treat it as an illustration of the general pattern, not an exact memory audit.



In [3]:
import sys

# Create a NumPy array, Python list, and tuple with the same elements
array = np.arange(1000, dtype=np.int64)
py_list = list(range(1000))
py_tuple = tuple(range(1000))

# Calculate memory usage
array_memory = array.nbytes
list_memory = sys.getsizeof(py_list) + sum(sys.getsizeof(item) for item in py_list)
tuple_memory = sys.getsizeof(py_tuple) + sum(sys.getsizeof(item) for item in py_tuple)

# Display the memory usage
memory_usage = {
    "NumPy Array (in bytes)": array_memory,
    "Python List (in bytes)": list_memory,
    "Python Tuple (in bytes)": tuple_memory
}

memory_usage


{'NumPy Array (in bytes)': 8000,
 'Python List (in bytes)': 36056,
 'Python Tuple (in bytes)': 36040}

In [4]:
# each element in the array is a 64-bit integer
array.dtype

dtype('int64')

Here, `int64` means each integer occupies 64 bits (8 bytes), so the array's element data take **1,000 × 8 = 8,000 bytes**. The list and tuple totals are larger because they also include container and object overhead, and the exact numbers can vary across Python versions and platforms.

You will work with dtypes throughout the rest of this chapter, and revisit shared memory in [Views and Copies](#views-and-copies).


### Numerical Operations and Timing

Many NumPy operations call compiled routines instead of running a Python loop, which can make them faster, but the speedup depends on the operation, dtype, array shape, and data size. NumPy is not automatically faster for every task, and vectorization does not by itself guarantee parallel execution.

The optional example below times a Python loop against `np.dot`. A **dot product** multiplies corresponding entries of two vectors and adds the results: the dot product of `[1, 2, 3]` and `[4, 5, 6]` is `1*4 + 2*5 + 3*6 = 32`. For 1D arrays, `np.dot(a, b)` computes the same result as `np.sum(a * b)`.

On a first read, focus on what each approach computes and how their measured times compare; you do not need to memorize the timing code itself. Matrix multiplication gets fuller treatment in [NumPy Intermediate](vectorized_numpy.ipynb).


The example uses reproducible random inputs and confirms the two approaches agree using `np.isclose`, which allows for tiny floating-point differences. `repeat` runs each approach several times and reports the fastest observed time per call. `lambda` just wraps each calculation so the timer can call it repeatedly.



In [5]:
def my_dot(a, b): 
    """
   Compute the dot product of two vectors
 
    Args:
      a (ndarray (n,)):  input vector 
      b (ndarray (n,)):  input vector with same dimension as a
    
    Returns:
      x (scalar): 
    """
    x=0
    for i in range(a.shape[0]):
        x = x + a[i] * b[i]
    return x

In [6]:
from timeit import repeat
benchmark_rng = np.random.default_rng(303)
a = benchmark_rng.random(20_000)
b = benchmark_rng.random(20_000)
print('Results agree:', np.isclose(np.dot(a, b), my_dot(a, b)))
for label, fn in [('np.dot', lambda: np.dot(a, b)), ('Python loop', lambda: my_dot(a, b))]:
    print(label, 'best seconds per call:', min(repeat(fn, number=3, repeat=3)) / 3)
del a, b


Results agree: True
np.dot best seconds per call: 1.0556735408802826e-06
Python loop best seconds per call: 0.001453763999355336


Use the measured times to compare the two approaches for this calculation on this input size; the ranking is not universal. Creating arrays, changing dtypes, and allocating temporary results can also shift the runtime of real workloads.



## Building Blocks: NumPy Arrays Fundamentals

###  Array Creation: Your Complete Toolkit

NumPy offers multiple ways to create arrays, each optimized for different scenarios.

#### From Existing Data

| **Method**         | **Purpose**                        | **Example Use Case**                |
|---------------------|------------------------------------|--------------------------------------|
| `np.array()`        | Convert lists/tuples to arrays     | Transform Python data structures     |
| `df.to_numpy()`     | Convert a Pandas DataFrame         | Bridge between Pandas and NumPy   

In [7]:
# 1️⃣ Creating arrays from existing data
print("1️⃣ FROM EXISTING DATA")
print("=" * 30)

# From Python lists
data_1d = [1, 2, 3, 4, 5]
arr_from_list = np.array(data_1d)
print(f"From list: {arr_from_list}")

# From nested lists (2D array)
data_2d = [[1, 2, 3], [4, 5, 6]]
arr_2d = np.array(data_2d)
print(f"2D array:\n{arr_2d}")

# From tuples
arr_from_tuple = np.array((10, 20, 30))
print(f"From tuple: {arr_from_tuple}")

1️⃣ FROM EXISTING DATA
From list: [1 2 3 4 5]
2D array:
[[1 2 3]
 [4 5 6]]
From tuple: [10 20 30]


The **`to_numpy()`** method is used to convert a pandas DataFrame into a NumPy array. 

In [8]:
df = pd.DataFrame({'Tickets': [120, 80, 150], 'Revenue': [1200, 960, 1500]},
                  index=[104, 101, 107])
array = df[['Tickets', 'Revenue']].to_numpy(dtype=float, copy=True)
print('Labeled DataFrame:')
print(df)
print('Numeric array:')
print(array)
print('DataFrame label 101:', df.loc[101, 'Tickets'])
print('Array position [1, 0]:', array[1, 0])


Labeled DataFrame:
     Tickets  Revenue
104      120     1200
101       80      960
107      150     1500
Numeric array:
[[ 120. 1200.]
 [  80.  960.]
 [ 150. 1500.]]
DataFrame label 101: 80
Array position [1, 0]: 80.0


`to_numpy()` returns an ndarray. Here we select two numeric columns in an explicit order and request a copy. The array contains neither the labels `[104, 101, 107]` nor the column names; preserve those separately if you need them later. Position `[1, 0]` refers to the second row and first selected column.


In [9]:
df.to_numpy(dtype=float)

array([[ 120., 1200.],
       [  80.,  960.],
       [ 150., 1500.]])

Mixed-type DataFrames can produce object arrays; selecting meaningful numeric columns first avoids many surprises. NumPy does not align by labels, so changing the row order of one operand can silently change a calculation. Next comes [Pandas Intermediate](data_types_in_pandas.ipynb), followed by [NumPy Intermediate](vectorized_numpy.ipynb).


#### Specialized Constructors

| **Method**            | **Creates**              | **When to Use**                              |
|------------------------|--------------------------|----------------------------------------------|
| `np.zeros(shape)`      | Array of zeros           | Initialize arrays, placeholders              |
| `np.ones(shape)`       | Array of ones            | Mathematical operations, masks               |
| `np.full(shape, val)`  | Array filled with a value | Default values, initialization               |
| `np.eye(n)`            | Identity matrix          | Linear algebra operations                    |
| `np.empty(shape)`      | Uninitialized array      | Allocate storage; fill every entry before reading it             |

In [10]:

print("\n2️⃣ SPECIALIZED CONSTRUCTORS")
print("=" * 30)
# Arrays of zeros and ones
zeros_3x3 = np.zeros((3, 3))
ones_2x4 = np.ones((2, 4))
full_matrix = np.full((2, 3), 7)  # Fill with custom value

print(f"Zeros (3x3):\n{zeros_3x3}")
print(f"Ones (2x4):\n{ones_2x4}")
print(f"Full of 7s:\n{full_matrix}")

# Identity matrix
identity = np.eye(3)
print(f"Identity matrix:\n{identity}")


2️⃣ SPECIALIZED CONSTRUCTORS
Zeros (3x3):
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]
Ones (2x4):
[[1. 1. 1. 1.]
 [1. 1. 1. 1.]]
Full of 7s:
[[7 7 7]
 [7 7 7]]
Identity matrix:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]


#### Sequential & Mathematical Arrays

| **Method**                | **Purpose**                 | **Best For**                           |
|----------------------------|-----------------------------|----------------------------------------|
| `np.arange(start, stop, step)` | Values with a specified step      | Index generation, iteration            |
| `np.linspace(start, stop, num)` | A specified number of evenly spaced values        | Plotting, function evaluation          |
| `np.logspace(start, stop, num)` | Logarithmically spaced      | Scientific computing, exponential data |


In [11]:
print("\n3️⃣ SEQUENTIAL ARRAYS")
print("=" * 30)

# Using arange (like Python's range)
sequence1 = np.arange(10)          # 0 to 9
sequence2 = np.arange(5, 15)       # 5 to 14
sequence3 = np.arange(0, 10, 2)    # 0, 2, 4, 6, 8

print(f"arange(10): {sequence1}")
print(f"arange(5, 15): {sequence2}")
print(f"arange(0, 10, 2): {sequence3}")

# Using linspace (evenly spaced floats)
linear = np.linspace(0, 1, 5)      # 5 points between 0 and 1
print(f"linspace(0, 1, 5): {linear}")


3️⃣ SEQUENTIAL ARRAYS
arange(10): [0 1 2 3 4 5 6 7 8 9]
arange(5, 15): [ 5  6  7  8  9 10 11 12 13 14]
arange(0, 10, 2): [0 2 4 6 8]
linspace(0, 1, 5): [0.   0.25 0.5  0.75 1.  ]


`arange` excludes its stop value. `linspace` includes the endpoint by default and is useful when the number of samples matters. Prefer `linspace` for a fixed number of floating-point samples because step rounding can affect `arange`. `np.logspace(0, 2, 3)` gives powers of ten from 10⁰ to 10². `empty()` does not initialize usable numeric values; use zeros when that is the intended starting value.


#### Loading from Files

| **Method**        | **File Type**          | **Features**                                  |
|-------------------|------------------------|-----------------------------------------------|
| `np.load()`       | NumPy binary `.npy`    | Fast saving/loading of arrays                 |
| `np.loadtxt()`    | Simple text files      | Fast, lightweight parsing                     |
| `np.genfromtxt()` | Complex text files     | Handles missing values and mixed data types   |

###  Understanding Array Attributes

Let us define a NumPy array in order to access its attributes:

In [12]:
numpy_ex = np.array([[1,2,3],[4,5,6]])
numpy_ex

array([[1, 2, 3],
       [4, 5, 6]])

In [13]:
type(numpy_ex)

numpy.ndarray


It is an **ndarray** type.  

You can explore the attributes and methods of `numpy_ex` by typing:

```python
numpy_ex.
```
and then pressing the *tab* key.

Some of the basic attributes of a NumPy array are the following:

#### [`ndim`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.ndim.html) 
Shows the number of dimensions (or axes) of the array.

In [14]:
numpy_ex.ndim

2

For a 2D array, assign meaning to both dimensions:

```text
                 axis 1: columns
                 0   1   2
axis 0: rows  0 [ 1   2   3 ]
              1 [ 4   5   6 ]
shape = (2, 3); ndim = 2; size = 6
```

A shape `(3,)` is a 1D array: it is neither a `(1, 3)` row matrix nor a `(3, 1)` column matrix.


#### [`shape`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.shape.html)
This is a tuple of integers indicating the size of the array in each dimension. For a matrix with *n* rows and *m* columns, the shape will be *(n,m)*. The length of the shape tuple is therefore the rank, or the number of dimensions, `ndim`.

In [15]:
numpy_ex.shape

(2, 3)

#### [`size`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.size.html) 
This is the total number of elements of the array, which is the product of the elements of shape.

In [16]:
numpy_ex.size

6

#### [`dtype`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.dtype.html) 
Unlike lists and tuples, NumPy arrays are designed to store elements of the same type, enabling more efficient memory usage and faster computations. The data type of the elements in a NumPy array can be accessed using the `.dtype` attribute

In [17]:
numpy_ex.dtype

dtype('int64')

##  Data Types and Memory Optimization

NumPy supports a wide range of data types, each with a defined memory size. The dtype determines representation, range, and precision. Choose it for the values and operations you need, then consider storage.  

### Common NumPy Data Types

| **Data Type**    | **Memory Size**            |
|------------------|-----------------------------|
| `np.int8`        | 1 byte                      |
| `np.int16`       | 2 bytes                     |
| `np.int32`       | 4 bytes                     |
| `np.int64`       | 8 bytes                     |
| `np.uint8`       | 1 byte                      |
| `np.uint16`      | 2 bytes                     |
| `np.uint32`      | 4 bytes                     |
| `np.uint64`      | 8 bytes                     |
| `np.float16`     | 2 bytes                     |
| `np.float32`     | 4 bytes                     |
| `np.float64`     | 8 bytes                     |
| `np.complex64`   | 8 bytes                     |
| `np.complex128`  | 16 bytes                    |
| `np.bool_`       | 1 byte                      |
| `np.bytes_`     | 1 byte per character        |
| `np.str_`    | 4 bytes per character       |
| `np.object_`     | References; referent memory is additional   |
| `np.datetime64`  | 8 bytes                     |
| `np.timedelta64` | 8 bytes                     |


💡 **Tip:** Choosing the right data type is crucial for **memory efficiency** and **performance**. For large datasets, using smaller types (e.g., `int16` instead of `int64`) can save significant memory.

A smaller integer dtype has a narrower range. Arithmetic can overflow, and casting can lose precision. Inspect `np.iinfo(np.int16)` or `np.finfo(np.float32)` before choosing a smaller type. Fixed-width strings can truncate longer values assigned later.


In [18]:
# Add practical examples for data type selection
print("📊 CHOOSING THE RIGHT DATA TYPE")
print("=" * 40)

# Memory efficiency example
large_numbers = np.array([1000, 2000, 3000], dtype=np.int16)  # Efficient for small ranges
small_numbers = np.array([1, 2, 3], dtype=np.int8)           # Very memory efficient

print(f"int16 array memory: {large_numbers.nbytes} bytes")
print(f"int8 array memory: {small_numbers.nbytes} bytes")

# Precision example  
high_precision = np.array([3.14159265359], dtype=np.float64)
low_precision = np.array([3.14159265359], dtype=np.float32)
print(f"float64 precision: {high_precision}")
print(f"float32 precision: {low_precision}")

📊 CHOOSING THE RIGHT DATA TYPE
int16 array memory: 6 bytes
int8 array memory: 3 bytes
float64 precision: [3.14159265]
float32 precision: [3.1415927]


### Type Promotion and Conversion

When creating an array from mixed inputs, NumPy infers a common dtype. This does not guarantee exact preservation of every value: integers may lose precision in floating-point storage, and later casts may truncate or overflow. Inspect the dtype and sample results. Numeric promotion and conversion to strings follow different rules.


**Numeric Upcasting**: If you mix integers and floats, NumPy will convert the entire array to floats.

In [19]:
arr = np.array([1, 2.5, 3])
print(arr.dtype)  

float64


**String Upcasting**: If you mix numbers and strings, NumPy will upcast all elements to strings.

In [20]:
arr = np.array([1, 'hello', 3.5])
print(arr.dtype)

<U32


"<U32" means: a Unicode string array where each element can hold up to 32 characters, using little-endian byte order.

##  Array Indexing and Slicing: Accessing Your Data

###  Array Indexing
Similar to Python lists, NumPy uses zero-based indexing, meaning the first element of an array is accessed using index `0`. You can use positive or negative indices to access elements


In [21]:
array = np.array([10, 20, 30, 40, 50])

print(array[0])  
print(array[4]) 
print(array[-1])  
print(array[-3])  

10
50
50
30


In multi-dimensional arrays, indices are separated by commas.
The first index refers to the row, and the second index refers to the column in a 2D array.

In [22]:

# 2D array (3 rows, 3 columns)
array_2d = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(array_2d)
print(array_2d[0, 1])  
print(array_2d[1, -1]) 
print(array_2d[-1, -1])  


[[1 2 3]
 [4 5 6]
 [7 8 9]]
2
6
9


### Array Slicing
Slicing is used to extract a sub-array from an existing array. 

The Syntax for slicing is `array[start:stop:step]`


In [23]:
array = np.array([10, 20, 30, 40, 50])

print(array[1:4])  
print(array[:3])  
print(array[2:])  
print(array[::2])  
print(array[::-1]) 


[20 30 40]
[10 20 30]
[30 40 50]
[10 30 50]
[50 40 30 20 10]


For slicing in Multi-Dimensional Arrays, use commas to separate slicing for different dimensions

In [24]:
###  Indexing & Slicing Quick Reference

# Extract a sub-array: elements from the first two rows and the first two columns
sub_array = array_2d[:2, :2]
print(sub_array)  

# Extract all rows for the second column
col = array_2d[:, 1]
print(col) 

# Extract the last two rows and last two columns
sub_array = array_2d[-2:, -2:]
print(sub_array)  


[[1 2]
 [4 5]]
[2 5 8]
[[5 6]
 [8 9]]


### Preserving a Dimension; Views and Copies {#views-and-copies}

This section introduces two separate ideas: keeping a result in rows-and-columns form, and deciding whether editing a selection changes the original array.

**1. Preserving a dimension means keeping the result in rows-and-columns form.**

Suppose we have:


In [ ]:
a = np.array([
    [10, 20, 30],
    [40, 50, 60],
    [70, 80, 90]
])


These selections get the same numbers, but return different shapes:

| Selection | Result | Shape |
|---|---|---|
| `a[:, 1]` | `[20, 50, 80]` | `(3,)`: a 1D array with 3 elements |
| `a[:, 1:2]` | `[[20], [50], [80]]` | `(3, 1)`: a 2D array with 3 rows and 1 column |

In both expressions, `:` means “all rows.”

- `1` selects the second column and **removes that dimension**.
- `1:2` selects columns starting at 1 and stopping before 2, so it **keeps the column dimension**, even though only one column remains.

A 1D array has no separate row or column axis. Keeping the result 2D matters when a later operation expects a column.

**2. Views and copies determine whether editing the selection changes the original.**

A basic slice creates a **view**: it gives you access to part of the original data.


In [ ]:
original = np.array([10, 20, 30, 40])
piece = original[1:3]          # [20, 30]

piece[0] = 99

print(original)
# [10, 99, 30, 40]


[10 99 30 40]


`piece[0]` and `original[1]` refer to the same stored value. Changing it through `piece` changes what you see in `original`.

Adding `.copy()` creates independent data:


In [ ]:
original = np.array([10, 20, 30, 40])
piece = original[1:3].copy()   # Independent [20, 30]

piece[0] = 99

print(piece)
# [99, 30]

print(original)
# [10, 20, 30, 40]


[99 30]
[10 20 30 40]


These ideas are independent: **shape describes how the result is organized; view versus copy describes whether it shares data with the original.** Both `a[:, 1]` and `a[:, 1:2]` are views, despite having different shapes.


The `step` parameter can be used to select elements at regular intervals.

In [26]:
# the step parameter in slicing
print(array[1:8:2]) 
print(array[::-2])  


[20 40]
[50 30 10]


### Combining Indexing and Slicing
You can combine indexing and slicing to extract specific elements or sub-arrays

In [27]:
# Create a 3D array
array_3d = np.array([[[1, 2, 3], [4, 5, 6]], [[7, 8, 9], [10, 11, 12]]])

# Select specific elements and slices
print(array_3d[0, :, 1])  # Output: [2 5] (second element from each row in the first sub-array)
print(array_3d[1, 1, :2])  # Output: [10 11] (first two elements in the last row of the second sub-array)


[2 5]
[10 11]


### Boolean Mask: Conditional Selection

####  Single Condition in NumPy

You can use **boolean arrays** to filter or select elements that meet a specific condition.

In [28]:
a = np.array([10, 20, 30, 40, 50])
mask = a > 25                # [False False  True  True  True]
filtered = a[mask]           # [30 40 50]
# one-liner:
a[a > 25]  # Output: [30 40 50]

array([30, 40, 50])

#### Combining Multiple Conditions in NumPy

Like in **pandas**, you can use **bitwise operators** with parentheses to combine multiple conditions:

`&` (AND), `|` (OR), `~` (NOT), `^` (XOR)

For example:

In [29]:
print(a[(a > 15) & (a < 45)] )     
print(a[(a < 20) | (a > 40)]  )   
print(a[~(a % 20 == 0)] )

[20 30 40]
[10 50]
[10 30 50]


###  Indexing & Slicing Quick Reference

| **Operation** | **Syntax** | **Example** | **Result** |
|---------------|------------|-------------|------------|
| Single element | `arr[i]` | `arr[2]` | Element at index 2 |
| Negative indexing | `arr[-i]` | `arr[-1]` | Last element |
| Basic slice | `arr[start:stop]` | `arr[1:4]` | Elements 1, 2, 3 |
| Step slice | `arr[start:stop:step]` | `arr[::2]` | Every 2nd element |
| 2D indexing | `arr[row, col]` | `arr[0, 1]` | Row 0, Column 1 |
| Boolean mask | `arr[condition]` | `arr[arr > 5]` | Elements > 5 |

## Advanced Selection Methods

Beyond basic indexing and slicing, NumPy provides vectorized tools for conditional logic and for locating extreme values — useful for cleaning data, engineering features, and answering "which one is the best/worst?" without writing a loop.


### Conditional Selection with `np.where()` and `np.select()`

Use these NumPy tools for fast, vectorized if/else logic on arrays.

- `np.where(condition, x, y)`: simple **two-way** branching, choosing element-wise between `x` and `y`.
- `np.where(condition)`: returns the **indices** where `condition` is `True` (no `x`/`y` given).
- `np.select([cond1, cond2, ...], [choice1, choice2, ...], default=...)`: clean **multi-branch** logic for 3+ cases.

### `np.where()`


In [30]:
# Let's create sample data for our examples
ages = np.array([22, 25, 19, 35, 28, 24, 31, 26, 23, 29])
print(f"Student ages: {ages}")

scores = np.array([95, 87, 76, 63, 89, 92, 45, 78, 85, 91])
print(f"Student scores: {scores}")

Student ages: [22 25 19 35 28 24 31 26 23 29]
Student scores: [95 87 76 63 89 92 45 78 85 91]


In [ ]:
# Replace values based on a condition: scores below 70 become 0
pass_fail = np.where(scores >= 70, scores, 0)
print("Original scores:", scores)
print("After np.where (failing scores -> 0):", pass_fail)

# Get the indices (and values) of high scorers
high_idx = np.where(scores > 85)[0]
high_vals = scores[high_idx]
print("\nIndices of scores above 85:", high_idx)
print("Values above 85:", high_vals)


📊 USING np.where()

1️⃣ Pass/Fail (70+ to pass):
Original: [95 87 76 63 89 92 45 78 85 91]
Result:   [95 87 76  0 89 92  0 78 85 91]

2️⃣ High Scorers (>85):
Indices: [0 1 4 5 9]
Values : [95 87 89 92 91]


- **Chainable**
  - You can **nest** multiple `np.where()` calls, but prefer `np.select` for **3+ branches**.

In [ ]:
# Nested np.where for multi-way replacement (letter grades)
letter_grades = np.where(scores >= 90, 'A',
                np.where(scores >= 80, 'B',
                np.where(scores >= 70, 'C',
                np.where(scores >= 60, 'D', 'F'))))
print("Letter grades:", letter_grades)

# Using np.where to find indices matching a condition
a_idx = np.where(letter_grades == 'A')[0]
f_idx = np.where(letter_grades == 'F')[0]
print("Indices of A's:", a_idx)
print("Indices of F's:", f_idx)



3️⃣ Indices of A's and F's:
A indices: [0 5 9]
F indices: [6]


While you can nest multiple `np.where()` calls for complex conditions, **`np.select()`** is preferred for **3+ branches** because it’s more readable and maintainable.

### `np.select`: Multi-way Conditional Logic

#### Syntax
```python
np.select(conditions, choices, default=...)
```

- `conditions`: list of boolean arrays (must be same shape or broadcastable)
- `choices`: list of result values/arrays (same length as `conditions`)
- `default`: value used where no condition is `True` (optional)

#### Example 1 — Letter grades (cleaner than nested `np.where()`)

In [ ]:
scores = np.array([95, 87, 76, 63, 89, 92, 45, 78, 85, 91])
conds   = [scores >= 90, scores >= 80, scores >= 70, scores >= 60]
choices = ['A',          'B',          'C',          'D']
grades  = np.select(conds, choices, default='F')
print("Original scores:", scores)
print("Letter grades with np.select():", grades)



4️⃣ Letter Grades with np.select(): ['A' 'B' 'C' 'D' 'B' 'A' 'F' 'C' 'B' 'A']
Original Scores: [95 87 76 63 89 92 45 78 85 91]


#### Example 2 — Numeric binning (labels)

In [ ]:
x = np.array([5, 12, 20, 33, 47])
conds = [x < 10, (x >= 10) & (x < 30), x >= 30]
choices = ['low_precision', 'mid_precision', 'high_precision']
buckets = np.select(conds, choices, default='unknown')

print(buckets)


['low_precision' 'mid_precision' 'mid_precision' 'high_precision'
 'high_precision']


**`np.select` and the `default` dtype gotcha**

If you omit `default=...`, `np.select` falls back to the integer `0`. When your `choices` are strings, NumPy cannot find a common dtype for string choices and an integer default, so it raises a `TypeError`.

**Example (raises an error)**
```python
import numpy as np

x = np.array([5, 12, 30])
conds   = [x < 10, (x >= 10) & (x < 20)]
choices = ['low', 'mid']

np.select(conds, choices)   # default is 0 -> TypeError (mixed str + int)
```

Always pass an explicit `default` that matches the dtype of your `choices`.


**Best Practices:**

- **Use `np.where()`** for simple binary conditions
- **Use `np.select()`** for 3+ conditions or complex logic
- **Always provide a default** to handle edge cases
- **Test conditions** to ensure they're mutually exclusive when needed

If several conditions are true, `np.select` uses the **first** matching condition. Put the most specific/highest threshold first when conditions overlap. The inputs to `where` are evaluated before selection; it is not a way to prevent invalid arithmetic in an unselected branch.


### Finding Minimum and Maximum Values and Positions {#min-max-search}

- `np.min` / `np.max` return the **values**; `np.argmin` / `np.argmax` return their **positions** (indices).
- Omit `axis` for a single global position; specify `axis` to get one position per row or column.
- When there is a tie, a flattened search returns only the **first** occurrence in row-major order. Use a boolean mask (e.g. `a == a.max()`) to find every tied position instead.
- These are NumPy positions, not pandas row labels.



In [35]:
a = np.array([4, 1, 9, 7])
i_min = np.argmin(a)   # 1
i_max = np.argmax(a)   # 2

print("min @ index", i_min, "value:", a[i_min]) 
print("max @ index", i_max, "value:", a[i_max])  

min @ index 1 value: 1
max @ index 2 value: 9


In [ ]:
# 2D array of student test scores (4 students x 4 tests)
scores = np.array([
    [85, 92, 78, 95],  # Student 0
    [88, 76, 91, 82],  # Student 1
    [95, 89, 84, 90],  # Student 2
    [72, 85, 79, 88]   # Student 3
])
print("Student test scores:")
print(scores)

# Flattened search (axis=None): a single global index
min_idx = np.argmin(scores)
max_idx = np.argmax(scores)
print("\nLowest score index (flattened):", min_idx)
print("Highest score index (flattened):", max_idx)

# Convert the flat index back to (row, col) coordinates
min_row, min_col = np.unravel_index(min_idx, scores.shape)
max_row, max_col = np.unravel_index(max_idx, scores.shape)
print(f"Minimum {scores[min_row, min_col]} is Student {min_row}, Test {min_col}")
print(f"Maximum {scores[max_row, max_col]} is Student {max_row}, Test {max_col}")


📊 Student Test Scores (4 students × 4 tests):
[[85 92 78 95]
 [88 76 91 82]
 [95 89 84 90]
 [72 85 79 88]]

1️⃣ FLATTENED VIEW (axis=None) - Single Index
Lowest score index (flattened):  12
Highest score index (flattened): 3

🔻 Minimum: 72 at position (3, 0)
   → Student 3, Test 0
🔺 Maximum: 95 at position (0, 3)
   → Student 0, Test 3



In [ ]:
# axis=0: compare down each column -> which STUDENT is best/worst on each test
min_student_per_test = np.argmin(scores, axis=0)
max_student_per_test = np.argmax(scores, axis=0)
print("Lowest-scoring student per test:", min_student_per_test)
print("Highest-scoring student per test:", max_student_per_test)

for test_num in range(scores.shape[1]):
    worst_student = min_student_per_test[test_num]
    best_student = max_student_per_test[test_num]
    print(f"  Test {test_num}: worst = Student {worst_student} ({scores[worst_student, test_num]}), "
          f"best = Student {best_student} ({scores[best_student, test_num]})")

# axis=1: compare across each row -> which TEST was each student's best/worst
min_test_per_student = np.argmin(scores, axis=1)
max_test_per_student = np.argmax(scores, axis=1)
print("\nWorst test per student:", min_test_per_student)
print("Best test per student:", max_test_per_student)

for student_num in range(scores.shape[0]):
    worst_test = min_test_per_student[student_num]
    best_test = max_test_per_student[student_num]
    print(f"  Student {student_num}: worst = Test {worst_test} ({scores[student_num, worst_test]}), "
          f"best = Test {best_test} ({scores[student_num, best_test]})")


2️⃣ AXIS=0 (down columns) - Which STUDENT performed best/worst?
Lowest scoring student per test:  [3 1 0 1]
Highest scoring student per test: [2 0 1 0]

Detailed breakdown:
  Test 0: Worst = Student 3 (72), Best = Student 2 (95)
  Test 1: Worst = Student 1 (76), Best = Student 0 (92)
  Test 2: Worst = Student 0 (78), Best = Student 1 (91)
  Test 3: Worst = Student 1 (82), Best = Student 0 (95)

3️⃣ AXIS=1 (across rows) - Which TEST gave each student their highest/lowest score?
Worst test per student:  [2 1 2 0]
Best test per student:   [3 2 0 3]

Detailed breakdown:
  Student 0: Worst = Test 2 (78), Best = Test 3 (95)
  Student 1: Worst = Test 1 (76), Best = Test 2 (91)
  Student 2: Worst = Test 2 (84), Best = Test 0 (95)
  Student 3: Worst = Test 0 (72), Best = Test 3 (88)



**Tips**

- `argmin`/`argmax` return **indices**, not values (use them to index back into the array).
- **Axis behavior**
  - `axis=None` (default): flattens the entire array and returns the **global** index.
  - `axis=0` (2D): compares **down the rows** within each column → returns **row indices per column**.
  - `axis=1` (2D): compares **across columns** within each row → returns **column indices per row**.
  - General ND: reduces along the specified axis; the **result shape equals the input shape with that axis removed**.
- For values at those positions:
  - Column-wise: `i = np.argmax(M, axis=0); vals = M[i, np.arange(M.shape[1])]`
  - Row-wise:    `j = np.argmax(M, axis=1); vals = M[np.arange(M.shape[0]), j]`
- Convert a flat index to coordinates with `np.unravel_index(idx, arr.shape)`.


In [38]:
print('Maximum value:', np.max(scores))
print('All positions tied for the maximum:')
print(np.argwhere(scores == np.max(scores)))
print('Minimum value:', np.min(scores))


Maximum value: 95
All positions tied for the maximum:
[[0 3]
 [2 0]]
Minimum value: 72


`argwhere` returns one coordinate row per match. Here the maximum is tied; a single `argmax` reports only its first position. These searches assume nonempty inputs with valid values. Ordinary reductions can propagate `NaN`; inspect missing data first, or use a justified `nanmin`/`nanargmin` policy and handle all-missing inputs explicitly.


### Finding the Top-n with `np.argsort()`

`argsort` returns positions that would sort the values; it does not return the sorted values themselves. Ascending order is the default. The `order` argument is for structured-array field names, not ascending versus descending.

Use `np.argsort(a, kind='stable')[:k]` for the smallest k values, preserving input order within ties. Reversing an ascending order also reverses its ties. For the small signed positive integers below, stable sorting of `-a` gives descending values while retaining input order among ties. Do not negate unsigned integers or the minimum representable signed integer without handling overflow first.

#### 1D arrays


In [39]:
a = np.array([3, 10, 7, 10])
k = 2

# Indices that would sort ascending
idx_asc = np.argsort(a, kind='stable')               # e.g., [0, 2, 1, 3]
print("Indices to sort ascending:", idx_asc)

# top=k indeces (ascending by value)
topk_idx_asc = idx_asc[:k]           # e.g., [0, 2]
print(f"Smallest-{k} indices:", topk_idx_asc)

# the values corresponding to the top-k indices (ascending)
topk_vals_asc = a[topk_idx_asc]
print(f"Smallest-{k} values:", topk_vals_asc)

# Top-k indices (descending by value)
topk_idx_des = idx_asc[-k:][::-1]         # e.g., [3, 1]
print(f"Top-{k} indices (descending by value):", topk_idx_des)

# the values corresponding to the top-k indices (descending)
topk_vals_des = a[topk_idx_des]
print(f"Top-{k} values (descending by value):", topk_vals_des)


Indices to sort ascending: [0 2 1 3]
Smallest-2 indices: [0 2]
Smallest-2 values: [3 7]
Top-2 indices (descending by value): [3 1]
Top-2 values (descending by value): [10 10]


Shortcut (numeric arrays):

In [40]:
topk_idx = np.argsort(-a, kind='stable')[:k]         # also descending indices
topk_vals = a[topk_idx]
print(topk_idx)
print(topk_vals)

[1 3]
[10 10]


#### 2D arrays (row-wise or column-wise)

In [41]:
A = np.array([[3, 7, 2],
              [5, 1, 9]])
k = 2

# Row-wise: get the top-k column indices for each row
row_idx_sorted = np.argsort(A, axis=1)          # shape (n_rows, n_cols)
topk_cols_per_row = row_idx_sorted[:, -k:][:, ::-1]

# Column-wise: get the top-k row indices for each column
col_idx_sorted = np.argsort(A, axis=0)
topk_rows_per_col = col_idx_sorted[-k:, :][::-1, :]

To retrieve the values:

In [42]:
# Row-wise values (gather with advanced indexing)
rows = np.arange(A.shape[0])[:, None]           # column vector of row indices
topk_vals_per_row = A[rows, topk_cols_per_row]
print("Top-k values per row:\n", topk_vals_per_row)

# Column-wise values
cols = np.arange(A.shape[1])[None, :]
topk_vals_per_col = A[topk_rows_per_col, cols]
print("Top-k values per column:\n", topk_vals_per_col)

Top-k values per row:
 [[7 3]
 [9 5]]
Top-k values per column:
 [[5 7 9]
 [3 1 2]]


## Array Operations: Mathematical Power at Scale

NumPy arrays support all standard arithmetic operators (`+`, `-`, `*`, `/`, `**`, `%`) and apply them **element-wise**, using compiled routines instead of a Python loop.

### Arithmetic Operations

When both operands are arrays of the **same shape**, each operator combines corresponding elements. When the shapes differ, NumPy uses **broadcasting** (covered below) to decide whether the operation is valid.

`*` multiplies corresponding elements; `@` performs matrix multiplication and follows different shape rules.


####  Array and Array

Let's explore **arithmetic operations between two arrays** through practical examples:

In [ ]:
# Sample data: sales figures for 3 stores x 4 quarters (in thousands)
store_sales_q1_q4 = np.array([[120, 150, 180, 200],  # Store A
                               [100, 130, 160, 190],  # Store B
                               [140, 110, 150, 170]]) # Store C

# Bonus amounts for each store and quarter
bonus_amounts = np.array([[10, 15, 20, 25],
                          [12, 18, 22, 28],
                          [8, 12, 18, 22]])

print("Quarterly sales:\n", store_sales_q1_q4)
print("Quarterly bonuses:\n", bonus_amounts)

# For compatibility with existing examples
arr1, arr2 = store_sales_q1_q4, bonus_amounts


🏪 Quarterly Sales (in thousands):
Store  Q1   Q2   Q3   Q4
A     [120 150 180 200]
B     [100 130 160 190]
C     [140 110 150 170]

💰 Quarterly Bonuses (in thousands):
Store  Q1   Q2   Q3   Q4
A     [10 15 20 25]
B     [12 18 22 28]
C     [ 8 12 18 22]


In [44]:
#Element-wise summation of arrays
arr1 + arr2

array([[130, 165, 200, 225],
       [112, 148, 182, 218],
       [148, 122, 168, 192]])

In [45]:
# Element-wise subtraction
arr2 - arr1

array([[-110, -135, -160, -175],
       [ -88, -112, -138, -162],
       [-132,  -98, -132, -148]])

In [46]:
# Element-wise multiplication
arr1 * arr2

array([[1200, 2250, 3600, 5000],
       [1200, 2340, 3520, 5320],
       [1120, 1320, 2700, 3740]])

#### Broadcasting: The Heart of NumPy Vectorization

**Broadcasting** is NumPy's mechanism for combining arrays of different shapes without explicitly repeating their values. You may still need to reshape an operand to express the intended axis.

- **Array and vector**: the vector is broadcast across rows or columns depending on its shape.
- **Array and scalar**: the scalar is applied to every element.

Broadcasting is what makes vectorization in NumPy so flexible.



##### Broadcasting Rules

Compare dimensions from the right. A pair is compatible when the sizes match or one is 1; a missing leading dimension acts as 1.

```text
Three stores, four products: (3, 4)
One price per product:          (4,)  -> result (3, 4)
One multiplier per store:    (3, 1)  -> result (3, 4)
Unshaped store multipliers:     (3,)  -> 4 and 3 conflict
```

Broadcasting can create a large result even when the inputs are small. Predict the output shape before calculating.


In [ ]:
# Broadcasting rules demonstration
# NumPy applies these rules internally; this helper just makes them explicit for study.

def check_broadcast_compatibility(shape1, shape2):
    """
    Check if two shapes are compatible for broadcasting
    """
    # Pad shorter shape with 1s on the left
    ndim1, ndim2 = len(shape1), len(shape2)
    max_ndim = max(ndim1, ndim2)
    
    shape1_padded = [1] * (max_ndim - ndim1) + list(shape1)
    shape2_padded = [1] * (max_ndim - ndim2) + list(shape2)
    
    result_shape = []
    compatible = True
    
    for i in range(max_ndim):
        dim1, dim2 = shape1_padded[i], shape2_padded[i]
        if dim1 == dim2:
            result_shape.append(dim1)
        elif dim1 == 1:
            result_shape.append(dim2)
        elif dim2 == 1:
            result_shape.append(dim1)
        else:
            compatible = False
            break
    
    return compatible, tuple(result_shape) if compatible else None

# Test different shape combinations
test_cases = [
    ((3, 4), (4,)),      # Compatible
    ((3, 4), (3, 1)),    # Compatible  
    ((3, 4), (2,)),      # Incompatible
    ((2, 3, 4), (4,)),   # Compatible
    ((2, 3, 4), (3, 4)), # Compatible
    ((2, 3, 4), (2, 1, 4)), # Compatible
    ((3, 4), (2, 3)),    # Incompatible
]

print("Broadcasting compatibility check:")
for shape1, shape2 in test_cases:
    compatible, result_shape = check_broadcast_compatibility(shape1, shape2)
    status = "compatible" if compatible else "incompatible"
    print(f"{str(shape1):>10} + {str(shape2):>10} -> {str(result_shape):>12} ({status})")


Broadcasting Compatibility Check:
--------------------------------------------------
    (3, 4) +       (4,) →       (3, 4) ✅ Compatible
    (3, 4) +     (3, 1) →       (3, 4) ✅ Compatible
    (3, 4) +       (2,) →          N/A ❌ Incompatible
 (2, 3, 4) +       (4,) →    (2, 3, 4) ✅ Compatible
 (2, 3, 4) +     (3, 4) →    (2, 3, 4) ✅ Compatible
 (2, 3, 4) +  (2, 1, 4) →    (2, 3, 4) ✅ Compatible
    (3, 4) +     (2, 3) →          N/A ❌ Incompatible


**Comprehensive Broadcasting Examples:**

In [ ]:
# Example 1: 2D array + 1D array (most common case)
arr_2d = np.array([[1, 2, 3, 4], 
                   [5, 6, 7, 8], 
                   [9, 10, 11, 12]])
arr_1d = np.array([10, 20, 30, 40])

result_1 = arr_2d + arr_1d
print("Example 1: (3,4) + (4,) ->", result_1.shape)
print(result_1)

# Example 2: 2D array + column vector
arr_col = np.array([[100], 
                    [200], 
                    [300]])  # shape (3, 1)

result_2 = arr_2d + arr_col
print("\nExample 2: (3,4) + (3,1) ->", result_2.shape)
print(result_2)

# Example 3: array + scalar (always broadcastable)
result_3 = arr_2d + 1000
print("\nExample 3: (3,4) + scalar ->", result_3.shape)
print(result_3)

# Example 4: 3D array + 2D array
arr_3d = np.random.randint(1, 10, (2, 3, 4))
arr_2d_broadcast = np.array([[1, 2, 3, 4]])  # shape (1, 4)

result_4 = arr_3d + arr_2d_broadcast
print("\nExample 4: (2,3,4) + (1,4) ->", result_4.shape)


=== BROADCASTING EXAMPLES ===

Example 1: (3,4) + (4,) broadcasting
2D array shape: (3, 4)
1D array shape: (4,)
Result shape: (3, 4)
Result:
[[11 22 33 44]
 [15 26 37 48]
 [19 30 41 52]]

Example 2: (3,4) + (3,1) broadcasting
2D array shape: (3, 4)
Column vector shape: (3, 1)
Result shape: (3, 4)
Result:
[[101 102 103 104]
 [205 206 207 208]
 [309 310 311 312]]

Example 3: (3,4) + scalar broadcasting
Result shape: (3, 4)
Result:
[[1001 1002 1003 1004]
 [1005 1006 1007 1008]
 [1009 1010 1011 1012]]

Example 4: (2,3,4) + (1,4) broadcasting
3D array shape: (2, 3, 4)
2D array shape: (1, 4)
Result shape: (2, 3, 4)
Broadcasting successful!



##### One Factor per Row or per Column {#broadcasting-by-meaning}

Write the meaning of each axis before choosing the factor shape. With stores in rows and products in columns, a 1D vector of product prices acts on columns. One multiplier per store must have shape `(number_of_stores, 1)`.


In [49]:
demo_units = np.array([[2, 3, 4], [5, 1, 2]])
demo_prices = np.array([10., 20., 5.])
demo_store_factors = np.array([1.0, 0.9])
demo_revenue = demo_units * demo_prices
print('Units, prices, result shapes:', demo_units.shape, demo_prices.shape, demo_revenue.shape)
print('Revenue in dollars:', demo_revenue)
print('One factor per store:', demo_revenue * demo_store_factors[:, None])


Units, prices, result shapes: (2, 3) (3,) (2, 3)
Revenue in dollars: [[20. 60. 20.]
 [50. 20. 10.]]
One factor per store: [[20. 60. 20.]
 [45. 18.  9.]]


**One incompatible example and one compatible example**


In [50]:
try:
    np.ones((3, 4)) + np.ones((3, 3))
except ValueError as error:
    print('Expected incompatible shapes:', error)

valid_result = np.ones((2, 3)) + np.ones((2, 1, 1))
print('Compatible (2, 3) + (2, 1, 1):', valid_result.shape)


Expected incompatible shapes: operands could not be broadcast together with shapes (3,4) (3,3) 
Compatible (2, 3) + (2, 1, 1): (2, 2, 3)


The first operation fails because the trailing sizes 4 and 3 differ. The second succeeds: pad `(2, 3)` to `(1, 2, 3)` and compare it with `(2, 1, 1)` to obtain `(2, 2, 3)`. A calculation can be valid yet have an unintended extra dimension, so inspect the result shape. See the [NumPy broadcasting guide](https://numpy.org/doc/stable/user/basics.broadcasting.html).


### Aggregate Functions: Statistical Summaries

Aggregate functions reduce array dimensions by applying statistical operations. The `axis` parameter controls the direction of aggregation.

#### Global Aggregation (All Elements)

| **Function** | **Purpose** | **Example** |
|--------------|-------------|-------------|
| `np.sum(arr)` | Sum all elements | `np.sum([[1,2],[3,4]]) → 10` |
| `np.mean(arr)` | Average of all elements | `np.mean([[1,2],[3,4]]) → 2.5` |
| `np.min(arr)` | Minimum value | `np.min([[1,2],[3,4]]) → 1` |
| `np.max(arr)` | Maximum value | `np.max([[1,2],[3,4]]) → 4` |
| `np.std(arr)` | Standard deviation | `np.std([[1,2],[3,4]]) → 1.118` |

#### Axis-Specific Aggregation

**Understanding Axes in 2D Arrays:**

- `axis=0`: **Down the rows** (column-wise aggregation) → Result has shape (n_cols,)
- `axis=1`: **Across the columns** (row-wise aggregation) → Result has shape (n_rows,)

**Memory tip**: "Axis 0 goes down, Axis 1 goes across"

- The selected axis is **removed** by a reduction, unless you pass `keepdims=True`. For a `(3, 4)` array, `sum(axis=0)` has shape `(4,)`, and `sum(axis=1)` has shape `(3,)`. Label the units in both cases.
- NumPy's `std` uses `ddof=0` by default, while pandas' `std` defaults to `ddof=1` — the two are not directly comparable without matching `ddof`.
- Ordinary NumPy reductions propagate `NaN`; `nanmean` and related `nan*` functions skip it instead, so report the observed count when you use them.



In [51]:
# Student grades: 3 students × 4 subjects (Math, Science, English, History)
# Create sample data for demonstration
array = np.array([[4, 7, 1, 3],
                  [5, 8, 2, 6], 
                  [9, 3, 5, 2]])

# Display the original array
print("Original Array:\n", array)

# Calculate the sum, mean, minimum, and maximum for the entire array
total_sum = np.sum(array)
mean_value = np.mean(array)
min_value = np.min(array)
max_value = np.max(array)

print(f"\nSum of all elements: {total_sum}")  
print(f"Mean of all elements: {mean_value}")  
print(f"Minimum value in the array: {min_value}")  
print(f"Maximum value in the array: {max_value}")  

# Calculate the sum, mean, minimum, and maximum along each row (axis=1)
row_sum = np.sum(array, axis=1)
row_mean = np.mean(array, axis=1)
row_min = np.min(array, axis=1)
row_max = np.max(array, axis=1)

print("\nSum along each row:", row_sum)  
print("Mean along each row:", row_mean)  
print("Minimum value along each row:", row_min)  
print("Maximum value along each row:", row_max)  

# Calculate the sum, mean, minimum, and maximum along each column (axis=0)
col_sum = np.sum(array, axis=0)
col_mean = np.mean(array, axis=0)
col_min = np.min(array, axis=0)
col_max = np.max(array, axis=0)

print("\nSum along each column:", col_sum)  
print("Mean along each column:", col_mean)  
print("Minimum value along each column:", col_min)  
print("Maximum value along each column:", col_max)  


Original Array:
 [[4 7 1 3]
 [5 8 2 6]
 [9 3 5 2]]

Sum of all elements: 55
Mean of all elements: 4.583333333333333
Minimum value in the array: 1
Maximum value in the array: 9

Sum along each row: [15 21 19]
Mean along each row: [3.75 5.25 4.75]
Minimum value along each row: [1 2 2]
Maximum value along each row: [7 8 9]

Sum along each column: [18 18  8 11]
Mean along each column: [6.         6.         2.66666667 3.66666667]
Minimum value along each column: [4 3 1 2]
Maximum value along each column: [9 8 5 6]


In [52]:
print('Sales per store:', demo_revenue.sum(axis=1))
print('Sales per product:', demo_revenue.sum(axis=0))
print('Row totals kept as columns:', demo_revenue.sum(axis=1, keepdims=True).shape)
missing_demo = np.array([1.0, np.nan, 3.0])
print('Ordinary mean:', np.mean(missing_demo))
print('Observed count:', np.isfinite(missing_demo).sum(), 'nanmean:', np.nanmean(missing_demo))


Sales per store: [100.  80.]
Sales per product: [70. 80. 30.]
Row totals kept as columns: (2, 1)
Ordinary mean: nan
Observed count: 2 nanmean: 2.0


**Pause and explain:** For a `(3, 4)` table, which axis should disappear to give one total for each store? Is the output a `(3,)` vector or a `(3, 1)` column when `keepdims=True`?


##  Array Reshaping: Transforming Data Dimensions

**Array reshaping** is essential for preparing data for different computational tasks. Many operations require specific array shapes:

- **Machine Learning**: Models often expect specific input dimensions (e.g., 2D for tabular data, 4D for images)

- **Matrix Operations**: Linear algebra operations require compatible shapes for multiplication.

- **Data Analysis**: Different analysis techniques may need data in specific formats

- **Broadcasting**: Reshaping enables efficient element-wise operations between arrays

💡 **Key Insight**: `reshape` preserves element count and reinterprets their arrangement. It does not establish that the new axes have a meaningful interpretation; `resize`, below, can change the count.

###  Core Reshaping Methods

#### `reshape()`: Change Shape, Preserve Element Count

`reshape()` returns an array with the requested shape, sharing memory when possible and copying when necessary. Do not assume independence; use `.copy()` if later edits should be isolated.

**Syntax:** `array.reshape(new_shape)`. The product of the new dimensions must equal `array.size`.


In [ ]:
# Original 1D array (12 elements)
original = np.arange(1, 13)
print("Original (1D):", original, "shape:", original.shape)

# Reshape to 3D array (2 x 2 x 3)
array_3d = original.reshape(2, 2, 3)
print("\n3D array (2x2x3):\n", array_3d)

# Reshape to different 2D matrices
matrix_3x4 = original.reshape(3, 4)
print("\nMatrix (3x4):\n", matrix_3x4)

matrix_4x3 = original.reshape(4, 3)
print("\nMatrix (4x3):\n", matrix_4x3)

matrix_2x6 = original.reshape(2, 6)
print("\nMatrix (2x6):\n", matrix_2x6)


📊 RESHAPE() DEMONSTRATIONS
Original (1D): [ 1  2  3  4  5  6  7  8  9 10 11 12]
Original shape: (12,)

3D Array (2×2×3):
[[[ 1  2  3]
  [ 4  5  6]]

 [[ 7  8  9]
  [10 11 12]]]
Shape: (2, 2, 3)

Matrix (3×4):
[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
Shape: (3, 4)

Matrix (4×3):
[[ 1  2  3]
 [ 4  5  6]
 [ 7  8  9]
 [10 11 12]]
Shape: (4, 3)

Matrix (2×6):
[[ 1  2  3  4  5  6]
 [ 7  8  9 10 11 12]]
Shape: (2, 6)


####  Smart Dimension Inference with `-1`

Use `-1` to let NumPy **automatically calculate** one dimension based on the array size and other specified dimensions.

In [54]:
#  Automatic Dimension Calculation (-1)

data = np.arange(24)  # 24 elements
print(f"Original data: {data}")
print(f"Total elements: {data.size}")

# Use -1 to let NumPy infer exactly one dimension (only one -1 per reshape)

# 4 rows, infer columns -> 4×6
result_1 = data.reshape(4, -1)

# infer rows, 3 columns -> 8×3
result_2 = data.reshape(-1, 3)

# 2 × 3 × ? -> 2×3×4
result_3 = data.reshape(2, 3, -1)

print("\n4×? becomes: 4×{} = {}".format(result_1.shape[1], result_1.shape))
print(result_1)

print("\n?×3 becomes: {}×3 = {}".format(result_2.shape[0], result_2.shape))
print(result_2)

print("\n2×3×? becomes: 2×3×{} = {}".format(result_3.shape[2], result_3.shape))
print(result_3)


Original data: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
Total elements: 24

4×? becomes: 4×6 = (4, 6)
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]
 [12 13 14 15 16 17]
 [18 19 20 21 22 23]]

?×3 becomes: 8×3 = (8, 3)
[[ 0  1  2]
 [ 3  4  5]
 [ 6  7  8]
 [ 9 10 11]
 [12 13 14]
 [15 16 17]
 [18 19 20]
 [21 22 23]]

2×3×? becomes: 2×3×4 = (2, 3, 4)
[[[ 0  1  2  3]
  [ 4  5  6  7]
  [ 8  9 10 11]]

 [[12 13 14 15]
  [16 17 18 19]
  [20 21 22 23]]]


💡 **How `-1` Works**: NumPy calculates the missing dimension using the formula:
```
missing_dimension = total_elements ÷ (product_of_known_dimensions)

```


⚠️ **Important**: You can only use `-1` for **one dimension** per reshape operation!

####  `flatten()` vs `ravel()`: Converting to 1D

Both convert an array to 1D, but they differ in **copy vs view**, **speed**, and **memory order** handling.

| **Method**   | **Returns**     | **Copy/View**                                  | **Speed (typical)** | **When to Use**                                    |
|--------------|------------------|-----------------------------------------------|---------------------|----------------------------------------------------|
| `flatten()`  | 1D `ndarray`     | **Always makes a copy**                        | Allocates a copy    | You want an independent array you can modify safely |
| `ravel()`    | 1D `ndarray`     | **View if possible**, else copy (not guaranteed) | May avoid a copy    | You want a 1D view for efficiency (and accept that edits may affect the original) |

**Reading order options** (both functions support `order=`):
- **`'C'` (default)**: Row-major (left→right, top→bottom)
- **`'F'`**: Column-major (top→bottom, left→right)
- **`'A'`**: `'F'` if the array is Fortran-contiguous, otherwise `'C'`
- **`'K'`**: As close as possible to the array’s **in-memory layout** (preserves current strides, useful after slicing)

> ⚠️ **Gotcha:** `ravel()` may return a **copy** if a view isn’t possible (e.g., non-contiguous slices or mismatched `order`). If you must ensure independence, use `flatten()` or call `.copy()` on the result of `ravel()`.


In [ ]:
# Original 2D array
original_2d = np.array([[1, 2, 3],
                        [4, 5, 6],
                        [7, 8, 9]])
print("Original array:\n", original_2d)

# flatten() always returns a copy
flat_copy = original_2d.flatten()

# ravel() returns a view when possible (else a copy)
flat_view = original_2d.ravel()

print("\nflatten() result:", flat_copy)
print("ravel()   result:", flat_view)

print("\nShares memory with original?")
print("flatten():", np.shares_memory(original_2d, flat_copy))
print("ravel()  :", np.shares_memory(original_2d, flat_view))


📊 FLATTEN() vs RAVEL() COMPARISON
Original array:
[[1 2 3]
 [4 5 6]
 [7 8 9]]

Results:
flatten() result: [1 2 3 4 5 6 7 8 9]
ravel()   result: [1 2 3 4 5 6 7 8 9]

Memory sharing checks:
flatten() shares memory with original? False
ravel()   shares memory with original? True


In [ ]:
from time import perf_counter

# Small example matrix
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])
print(f"Matrix:\n{matrix}")

# Reading order: C (row-major) vs F (column-major)
c_order = matrix.flatten(order='C')  # left->right, top->bottom
f_order = matrix.flatten(order='F')  # top->bottom, left->right

print(f"\nC order (row-major): {c_order}")   # [1 2 3 4 5 6]
print(f"F order (col-major): {f_order}")     # [1 4 2 5 3 6] 

print("\nPerformance check (1000x1000 array):")
large_array = np.random.rand(1000, 1000)

# Simple timing helper: take the best of a few runs to reduce noise
def best_time(fn, reps=5):
    times = []
    for _ in range(reps):
        t0 = perf_counter()
        _ = fn()
        times.append(perf_counter() - t0)
    return min(times)

flatten_time = best_time(lambda: large_array.flatten(order='K'))
ravel_time   = best_time(lambda: large_array.ravel(order='K'))

print(f"flatten(): {flatten_time:.6f} seconds")
print(f"ravel()  : {ravel_time:.6f} seconds")
print('Timings describe this contiguous input; the two functions have different copy guarantees.')



📖 READING ORDER DEMONSTRATIONS
Matrix:
[[1 2 3]
 [4 5 6]]

C order (row-major): [1 2 3 4 5 6]
F order (col-major): [1 4 2 5 3 6]

⚡ Performance Test (1000×1000 array):
flatten(): 0.000106 seconds
ravel()  : 0.000000 seconds
Timings describe this contiguous input; the two functions have different copy guarantees.


####  `resize()` — Destructive Reshaping with Size Changes

`ndarray.resize(new_shape, refcheck=True)` changes an array **in place**. It’s powerful, but risky:

- ✅ **Can change the total number of elements** (unlike `reshape`)
- ✅ **Shrinking truncates** data
- ✅ **Expanding fills with zeros** (for numeric dtypes)
- ✅ **In-place / destructive**: permanently modifies the original array
- ⚠️ **Safety check**: raises `ValueError` if the array **references/is referenced**; do not bypass this check for shared arrays
- ❌ Don’t confuse with `np.resize(a, new_shape)` which **returns a new array** and **repeats elements** to fill


In [ ]:
# Example 1: expanding in place (pads with zeros)
array1 = np.array([1, 2, 3, 4], dtype=int)
original_id = id(array1)
print(f"Before resize: {array1}  shape={array1.shape}")
array1.resize(2, 4)  # 4 -> 8 elements; pads with zeros for numeric dtypes
print(f"After  resize:\n{array1}  shape={array1.shape}")
print("Same Python object?", id(array1) == original_id)

# Example 2: shrinking in place (truncates data)
array2 = np.array([[10, 20, 30, 40],
                   [50, 60, 70, 80]], dtype=int)
print(f"\nBefore shrinking:\n{array2}  shape={array2.shape}")
array2.resize(1, 3)  # 8 -> 3 elements; truncates
print(f"After  resize:\n{array2}  shape={array2.shape}")

# Example 3: reshape is the safe, non-destructive alternative
array3 = np.array([1, 2, 3, 4, 5, 6], dtype=int)
try:
    array3.reshape(2, 4)  # 6 -> 8 elements: invalid
except ValueError as e:
    print(f"\nreshape(2, 4) error: {e}")

safe_reshape = array3.reshape(2, 3)  # 6 -> 6 elements: valid
print(f"Safe reshape (2, 3):\n{safe_reshape}")
print("Original unchanged:", array3)

# Bonus: np.resize returns a NEW array, repeating elements if needed
a = np.array([1, 2, 3])
b = np.resize(a, (2, 5))
print(f"\nnp.resize(a, (2, 5)):\n{b}")
print("Original a unchanged:", a)
print("Object identity does not prove the underlying data buffer stayed in place.")


🔧 RESIZE() OPERATIONS

[Example 1] Expand array in-place to shape (2, 4)
Before resize: [1 2 3 4]  | shape=(4,)  | id=4660230448
After  resize: 
[[1 2 3 4]
 [0 0 0 0]]  | shape=(2, 4)  | id=4660230448
Same Python array object? True

[Example 2] Shrink array in-place to shape (1, 3) — data is truncated
Before shrinking:
[[10 20 30 40]
 [50 60 70 80]]  | shape=(2, 4)
After  resize:
[[10 20 30]]  | shape=(1, 3)

[Example 3] Using reshape (safe, returns a new view/copy without changing size)
Original: [1 2 3 4 5 6]  | shape=(6,)  | id=4660231504
reshape(2, 4) error: cannot reshape array of size 6 into shape (2,4)
Safe reshape (2, 3):
[[1 2 3]
 [4 5 6]]  | shape=(2, 3)
Original unchanged: [1 2 3 4 5 6]  | shape=(6,)  | id=4660231504

[Bonus] np.resize returns a new array (does not modify the original)
Original a: [1 2 3]  | id=4660428848
np.resize(a, (2,5)):
[[1 2 3 1 2]
 [3 1 2 3 1]]  | id=4660049072
Original a unchanged: [1 2 3]
Object identity does not prove the underlying data buffer st

####  `transpose()` and `.T`: Matrix Transposition

**Transposition** flips an array along its diagonal—**rows become columns** (and vice versa). It’s fundamental for linear algebra and data manipulation.

**Ways to transpose**

- **`.T`**: Shorthand for transposing a 2D array. For N-D arrays, it **reverses the axis order**.
- **`np.transpose(a, axes=None)`**: Function form. With `axes`, you can specify any axis permutation.
- **`a.transpose(*axes)`**: Method form (equivalent to `np.transpose(a, axes=...)`).

> ℹ️ All of the above return a **view** (no copy) when possible.

**Common applications**

- **Matrix multiplication**: Make shapes compatible (e.g., use `A.T @ B` when `A` is `(m, n)` and `B` is `(m, k)`).
- **Data analysis**: Exchange which variable is represented by each axis; this does not necessarily copy the data buffer.
- **Neural networks**: Align weight tensors and activations.
- **Statistics**: Work with covariance/correlation matrices.

In [ ]:
# 2D matrix transposition
matrix = np.array([[1, 2, 3, 4],
                   [5, 6, 7, 8]])
print(f"Original (2x4):\n{matrix}")

# Three equivalent ways to transpose
method1 = matrix.T
method2 = matrix.transpose()
method3 = np.transpose(matrix)
print(f"\nTransposed (4x2):\n{method1}")
print("All methods equal?", np.array_equal(method1, method2) and np.array_equal(method2, method3))

# Matrix multiplication example
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])
print(f"\nA @ B:\n{A @ B}")
print(f"A @ B.T:\n{A @ B.T}")
print(f"A.T @ B:\n{A.T @ B}")

# Real-world example: reorganizing student grades by subject instead of by student
students = ["Alice", "Bob", "Charlie"]
subjects = ["Math", "Stats", "CS"]

students_data = np.array([
    [85, 92, 78],   # Alice
    [88, 76, 90],   # Bob
    [91, 89, 84],   # Charlie
])
print("\nBy student (rows = students):")
for i, student in enumerate(students):
    print(f"{student}: {students_data[i]}")

subjects_data = students_data.T
print("\nBy subject (rows = subjects):")
for i, subject in enumerate(subjects):
    print(f"{subject}: {subjects_data[i]}")


🔄 TRANSPOSE OPERATIONS
Original (2×4):
[[1 2 3 4]
 [5 6 7 8]]
Shape: (2, 4)

Transposed (4×2) — All methods identical:
[[1 5]
 [2 6]
 [3 7]
 [4 8]]
Shape: (4, 2)
All equal? True

🎯 Matrix Multiplication Example:
A (2×2): 
[[1 2]
 [3 4]]
B (2×2): 
[[5 6]
 [7 8]]
A × B:
[[19 22]
 [43 50]]
A × B.T:
[[17 23]
 [39 53]]
A.T × B:
[[26 30]
 [38 44]]

📊 PRACTICAL EXAMPLE: Student Grades
By Students (rows = students):
Alice: [85 92 78]
Bob: [88 76 90]
Charlie: [91 89 84]

By Subjects (rows = subjects):
Math: [85 88 91]
Stats: [92 76 89]
CS: [78 90 84]


In [59]:
vector = np.array([1, 2, 3])
print('1D transpose:', vector.T.shape)
print('Explicit row:', vector[None, :].shape)
print('Explicit column:', vector[:, None].shape)


1D transpose: (3,)
Explicit row: (1, 3)
Explicit column: (3, 1)


###  Reshaping Quick Reference

| **Method** | **Purpose** | **Memory** | **Size Change** | **In-Place** |
|------------|-------------|------------|-----------------|---------------|
| `reshape()` | Change dimensions | View (when possible) | ❌ No | ❌ No |
| `flatten()` | Convert to 1D | Copy | ❌ No | ❌ No |
| `ravel()` | Convert to 1D | View (when possible) | ❌ No | ❌ No |
| `resize()` | Change dimensions & size | In-place | ✅ Yes | ✅ Yes |
| `transpose()` / `.T` | Flip dimensions | View | ❌ No | ❌ No |

**Best Practices**:

- Use **`reshape()`** for safe dimension changes
- Use **`ravel()`** for fast 1D conversion
- Use **`flatten()`** when you need an independent copy
- **Avoid `resize()`** unless you specifically need to change array size
- Use **`.T`** for simple 2D matrix transposition

In [ ]:
# Simulate image data (height x width x channels)
image_data = np.random.randint(0, 256, (64, 64, 3), dtype=np.uint8)
print(f"Original image shape: {image_data.shape} (H, W, C)")
print(f"Memory usage: {image_data.nbytes:,} bytes")

# 1. Flatten for machine learning (each pixel as an RGB feature row)
flat_features = image_data.reshape(-1, 3)
print(f"\nML features: {flat_features.shape} (pixels x RGB)")

# 2. Batch processing format (batch x height x width x channels)
batch_format = image_data.reshape(1, 64, 64, 3)
print(f"Batch format: {batch_format.shape} (N, H, W, C)")

# 3. Channel-first format (channels x height x width) - PyTorch style
channels_first = image_data.transpose(2, 0, 1)
print(f"Channels first: {channels_first.shape} (C, H, W)")

# 4. Grayscale conversion (average the RGB channels)
grayscale = np.mean(image_data, axis=2, keepdims=True)
print(f"Grayscale: {grayscale.shape} (H, W, 1)")

# 5. Thumbnail via downsampling (a view, not a resized copy)
thumbnail = image_data[::8, ::8, :]
print(f"Thumbnail: {thumbnail.shape} (downsampled)")
print(f"Thumbnail bytes vs original: {thumbnail.nbytes:,} vs {image_data.nbytes:,}")
print("The thumbnail is a view: its nbytes does not imply the original buffer was freed.")


🖼️ PRACTICE: IMAGE DATA RESHAPING
Original image shape: (64, 64, 3) (H×W×C)
Total pixels: 4096
Memory usage: 12,288 bytes

🔄 Common Computer Vision Reshaping:
ML features: (4096, 3) (pixels×RGB)
Batch format: (1, 64, 64, 3) (N×H×W×C)
Channels first: (3, 64, 64) (C×H×W)
Grayscale: (64, 64, 1) (H×W×1)
Thumbnail: (8, 8, 3) (downsampled)

💾 Memory efficiency:
Original: 12,288 bytes
Thumbnail: 192 bytes
Reduction: 64.0× smaller
The thumbnail is a view: its nbytes does not imply the original buffer was freed.


##  Array Concatenation: Joining Arrays Together

**Array concatenation** combines multiple arrays into a single array. This is essential for:

- **Data merging**: Combining datasets from different sources
- **Batch processing**: Joining results from parallel computations  
- **Feature engineering**: Combining different feature sets
- **Time series**: Appending new data to existing sequences

💡 **Key Principle**: Arrays must have **compatible shapes** along all axes except the concatenation axis.

###  Understanding Axes in Concatenation

Before diving into methods, let's understand how axes work:

- **axis=0**: Concatenate **vertically** (stack rows) → increases number of rows
- **axis=1**: Concatenate **horizontally** (stack columns) → increases number of columns  
- **axis=2**: For 3D+ arrays, concatenate along depth dimension

💡 **Memory Tip**: Think of axis numbers as the dimension that **grows** during concatenation.

###  `np.concatenate()`: The Universal Joiner

`np.concatenate()` is the **most flexible** concatenation function. It can join arrays along any axis with fine-grained control.

**Syntax**: `np.concatenate((array1, array2, ...), axis=0)`

**Requirements**:

- All arrays must have the **same number of dimensions**
- Shapes must match along **all axes except the concatenation axis**

**Advantages**:

- Works with **any number of arrays**
- Can specify **any axis** for concatenation
- Collecting inputs and joining once can avoid repeated growth

####  Comprehensive Concatenation Examples

Let's explore concatenation with practical, real-world examples:

In [ ]:
# Semester 1 scores (students x subjects)
semester1 = np.array([[85, 92, 78],    # Alice: Math, Science, English
                      [88, 76, 91],    # Bob
                      [95, 89, 84]])   # Carol

# Semester 2 scores (same students, same subjects)
semester2 = np.array([[87, 89, 82],
                      [91, 78, 89],
                      [93, 92, 88]])

students = ['Alice', 'Bob', 'Carol']
subjects = ['Math', 'Science', 'English']

print("Semester 1 scores:")
for i, student in enumerate(students):
    print(f"{student:6}: {semester1[i]}")
    
print("\nSemester 2 scores:")
for i, student in enumerate(students):
    print(f"{student:6}: {semester2[i]}")


📚 STUDENT SCORE CONCATENATION
Semester 1 Scores:
Alice : [85 92 78]
Bob   : [88 76 91]
Carol : [95 89 84]

Semester 2 Scores:
Alice : [87 89 82]
Bob   : [91 78 89]
Carol : [93 92 88]


In [ ]:
# Add new students to the class
new_students = np.array([[82, 85, 79],    # David: Math, Science, English
                         [90, 88, 93]])   # Emma

# Concatenate along axis=0 (add rows = add students)
expanded_class = np.concatenate((semester1, new_students), axis=0)
print(f"Class size: {semester1.shape[0]} -> {expanded_class.shape[0]} students")

all_students = students + ['David', 'Emma']
for i, student in enumerate(all_students):
    print(f"{student:6}: {expanded_class[i]}")



🔽 AXIS=0 CONCATENATION (Vertical Stacking)
Original class size: 3 students
New students added: 2 students
Total class size: 5 students

Expanded class scores (5 students × 3 subjects):
[[85 92 78]
 [88 76 91]
 [95 89 84]
 [82 85 79]
 [90 88 93]]
Alice : [85 92 78]
Bob   : [88 76 91]
Carol : [95 89 84]
David : [82 85 79]
Emma  : [90 88 93]


In [ ]:
# Add new subjects (History and Art scores)
new_subjects_scores = np.array([[80, 92],    # Alice: History, Art
                                [85, 88],    # Bob
                                [91, 95]])   # Carol

# Concatenate along axis=1 (add columns = add subjects)
expanded_subjects = np.concatenate((semester1, new_subjects_scores), axis=1)
print(f"Subjects: {semester1.shape[1]} -> {expanded_subjects.shape[1]}")

all_subjects = subjects + ['History', 'Art']
print(f"Columns: {all_subjects}")
for i, student in enumerate(students):
    print(f"{student:6}: {expanded_subjects[i]}")



➡️ AXIS=1 CONCATENATION (Horizontal Stacking)
Original subjects: 3 subjects
New subjects added: 2 subjects
Total subjects: 5 subjects

Expanded scores (3 students × 5 subjects):
Subjects: ['Math', 'Science', 'English', 'History', 'Art']
[[85 92 78 80 92]
 [88 76 91 85 88]
 [95 89 84 91 95]]
Alice : [85 92 78 80 92]
Bob   : [88 76 91 85 88]
Carol : [95 89 84 91 95]


####  Visual Understanding of Concatenation

Here's how axis=0 and axis=1 concatenation work visually:

```
AXIS=0 (Vertical - Stack Rows):
┌─────────┐     ┌─────────┐     ┌─────────┐
│ Array 1 │  +  │ Array 2 │  =  │ Array 1 │
│ [1,2,3] │     │ [7,8,9] │     │ [1,2,3] │
│ [4,5,6] │     └─────────┘     │ [4,5,6] │
└─────────┘                     │ [7,8,9] │
                                └─────────┘

AXIS=1 (Horizontal - Stack Columns):
┌─────────┐  +  ┌─────┐  =  ┌───────────┐
│ Array 1 │     │Array│     │  Combined │
│ [1,2,3] │     │ [7] │     │ [1,2,3,7] │
│ [4,5,6] │     │ [8] │     │ [4,5,6,8] │
└─────────┘     └─────┘     └───────────┘
```

💡 **Rule of Thumb**: 

- **axis=0** → **More rows** (vertical growth)
- **axis=1** → **More columns** (horizontal growth)

#### ⚠️ Common Concatenation Errors and Solutions

Let's explore what happens when shapes don't match and how to fix it:

In [ ]:
# Shape mismatch example
original_2d = np.array([[1, 2, 3], 
                        [4, 5, 6]])
problem_1d = np.array([7, 8, 9])

print("2D array shape:", original_2d.shape)
print("1D array shape:", problem_1d.shape)
print("Different number of dimensions, so concatenation will fail.")


⚠️  CONCATENATION ERROR DEMONSTRATION
Original 2D array shape: (2, 3)
Original 2D array:
[[1 2 3]
 [4 5 6]]

Problematic 1D array shape: (3,)
Problematic 1D array: [7 8 9]

🚫 Why this fails:
   2D array: (2, 3) (2 dimensions)
   1D array: (3,) (1 dimension)
   ❌ Different number of dimensions!


In [ ]:
try:
    result = np.concatenate((original_2d, problem_1d), axis=0)
    print("Success!")
except ValueError as e:
    print("Error:", e)



🧪 Attempting concatenation (this will fail):
❌ Error: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s) and the array at index 1 has 1 dimension(s)
   → Arrays have different numbers of dimensions!


####  Solutions to Shape Mismatch Problems

When arrays don't have compatible shapes, here are the most common fixes:

In [ ]:
# Reshape the 1D array to add a missing dimension
print("Original problematic shape:", problem_1d.shape)

# Add a row dimension -> (1, 3)
solution_1 = problem_1d.reshape(1, 3)
print("Reshaped to row:", solution_1.shape, solution_1)

# Add a column dimension -> (3, 1)
solution_2 = problem_1d.reshape(3, 1)
print("Reshaped to column:", solution_2.shape)
print(solution_2)

# Using np.newaxis (equivalent, more explicit)
solution_3 = problem_1d[np.newaxis, :]  # same as reshape(1, 3)
solution_4 = problem_1d[:, np.newaxis]  # same as reshape(3, 1)
print("Row via newaxis:", solution_3.shape)
print("Column via newaxis:", solution_4.shape)


✅ SOLUTION 1: Reshape 1D → 2D
Original problematic shape: (3,)
Reshaped to row: (1, 3)
Reshaped array:
[[7 8 9]]

Reshaped to column: (3, 1)
Reshaped array:
[[7]
 [8]
 [9]]

Using newaxis:
Row format: (1, 3) → [[7 8 9]]
Col format: (3, 1) → 
[[7]
 [8]
 [9]]


#### ✅ Successful Concatenation After Reshaping

In [ ]:
# Now concatenation works with the reshaped array
fixed_array = problem_1d.reshape(1, 3)
success_result = np.concatenate((original_2d, fixed_array), axis=0)
print("Result shape:", success_result.shape)
print(success_result)
print("\n(2,3) + (3,) was incompatible; (2,3) + (1,3) -> (3,3) works.")


🎉 SUCCESSFUL CONCATENATION
Original 2D: (2, 3)
Fixed 1D→2D: (1, 3)

✅ Concatenation successful!
Result shape: (3, 3)
Result:
[[1 2 3]
 [4 5 6]
 [7 8 9]]

📊 What happened:
   • Started with (2,3) + (3,) → ❌ Incompatible
   • Reshaped to (2,3) + (1,3) → ✅ Compatible
   • Final result: (3,3) array


####  Multiple Array Concatenation

`np.concatenate()` can join more than two arrays at once:

In [ ]:
# Quarterly sales data (stores x products)
q1_sales = np.array([[100, 150], [120, 180]])
q2_sales = np.array([[110, 160], [130, 190]])
q3_sales = np.array([[105, 155], [125, 185]])
q4_sales = np.array([[115, 165], [135, 195]])

# Concatenate all quarters horizontally (timeline view: stores x Q1-Q4 products)
yearly_timeline = np.concatenate((q1_sales, q2_sales, q3_sales, q4_sales), axis=1)
print("Horizontal timeline shape:", yearly_timeline.shape)
print(yearly_timeline)

# Stack all quarters vertically (all store-quarters x products)
yearly_stack = np.concatenate((q1_sales, q2_sales, q3_sales, q4_sales), axis=0)
print("\nVertical stack shape:", yearly_stack.shape)
print(yearly_stack)


🔗 JOINING MULTIPLE ARRAYS
Quarterly Sales Data (Stores × Products):
Q1: 
[[100 150]
 [120 180]]
Q2: 
[[110 160]
 [130 190]]
Q3: 
[[105 155]
 [125 185]]
Q4: 
[[115 165]
 [135 195]]

Horizontal Timeline (Stores × Q1-Q4 Products):
Shape: (2, 8)
[[100 150 110 160 105 155 115 165]
 [120 180 130 190 125 185 135 195]]

Vertical Stack (All Store-Quarters × Products):
Shape: (8, 2)
[[100 150]
 [120 180]
 [110 160]
 [130 190]
 [105 155]
 [125 185]
 [115 165]
 [135 195]]

📈 Analysis:
Timeline view: 2 stores × 8 data points
Stacked view: 8 observations × 2 products


### Specialized Stacking Functions

| Function | Meaning |
|---|---|
| `np.vstack((A, B))` | Treat 1D inputs as rows, then join along axis 0 |
| `np.hstack((A, B))` | Join 1D inputs along axis 0; higher-dimensional inputs along axis 1 |
| `np.dstack((A, B))` | Promote inputs to at least 3D, then join along axis 2 |
| `np.column_stack((a, b))` | Treat 1D inputs as columns |
| `np.stack((A, B), axis=0)` | Create a new axis; all inputs must have the same shape |

Concatenation extends an existing axis; stacking creates a new one. Matching shapes do not guarantee matching people/products: keep row and column order consistent yourself.


In [ ]:
# Sample arrays for demonstration
array_a = np.array([[1, 2], [3, 4]])
array_b = np.array([[5, 6], [7, 8]])

# Vertical stacking (more rows)
vstack_result = np.vstack((array_a, array_b))
concat_v_result = np.concatenate((array_a, array_b), axis=0)
print("vstack == concatenate(axis=0)?", np.array_equal(vstack_result, concat_v_result))
print(vstack_result)

# Horizontal stacking (more columns)
hstack_result = np.hstack((array_a, array_b))
concat_h_result = np.concatenate((array_a, array_b), axis=1)
print("\nhstack == concatenate(axis=1)?", np.array_equal(hstack_result, concat_h_result))
print(hstack_result)

# Depth stacking (3D)
dstack_result = np.dstack((array_a, array_b))
print("\ndstack shape:", dstack_result.shape)
print("First slice:\n", dstack_result[:, :, 0])
print("Second slice:\n", dstack_result[:, :, 1])

# 1D arrays: vstack, hstack, and column_stack all treat them differently
vec1 = np.array([1, 2, 3])
vec2 = np.array([4, 5, 6])
print("\nvstack (treats 1D as rows):\n", np.vstack((vec1, vec2)))
print("hstack (concatenates):", np.hstack((vec1, vec2)))
print("column_stack (treats 1D as columns):\n", np.column_stack((vec1, vec2)))


🔗 STACKING FUNCTIONS COMPARISON
Array A:
[[1 2]
 [3 4]]
Array B:
[[5 6]
 [7 8]]

📊 VERTICAL STACKING (More Rows)
-----------------------------------
np.vstack():
[[1 2]
 [3 4]
 [5 6]
 [7 8]]
Equivalent concatenate(axis=0):
[[1 2]
 [3 4]
 [5 6]
 [7 8]]
Results identical: True

➡️  HORIZONTAL STACKING (More Columns)
--------------------------------------
np.hstack():
[[1 2 5 6]
 [3 4 7 8]]
Equivalent concatenate(axis=1):
[[1 2 5 6]
 [3 4 7 8]]
Results identical: True

🔺 DEPTH STACKING (3D Arrays)
-------------------------
np.dstack() shape: (2, 2, 2)
First 'slice':
[[1 2]
 [3 4]]
Second 'slice':
[[5 6]
 [7 8]]

🧮 SPECIAL: 1D Array Handling
------------------------------
Vector 1: [1 2 3]
Vector 2: [4 5 6]
vstack (treats as rows):
[[1 2 3]
 [4 5 6]]
hstack (concatenates): [1 2 3 4 5 6]
column_stack (treats as cols):
[[1 4]
 [2 5]
 [3 6]]


In [70]:
stacked_semesters = np.stack((semester1, semester2), axis=0)
print('Semester x student x subject:', stacked_semesters.shape)


Semester x student x subject: (2, 3, 3)


###  Advanced Concatenation Techniques

In [71]:
# Optional: compare equal concatenations on modest fixed inputs.
from timeit import repeat
concat_rng = np.random.default_rng(303)
large_arrays = [concat_rng.random((100, 20)) for _ in range(10)]
def growing_concat():
    result = large_arrays[0]
    for part in large_arrays[1:]:
        result = np.concatenate((result, part), axis=0)
    return result

def batch_concat():
    return np.concatenate(large_arrays, axis=0)

def preallocated_concat():
    result = np.empty((1000, 20))
    for i, part in enumerate(large_arrays):
        result[i * 100:(i + 1) * 100] = part
    return result

np.testing.assert_array_equal(growing_concat(), batch_concat())
np.testing.assert_array_equal(preallocated_concat(), batch_concat())
for label, fn in [('Repeated growth', growing_concat), ('One concatenate', batch_concat), ('Preallocated', preallocated_concat)]:
    print(label, 'best seconds per call:', min(repeat(fn, number=3, repeat=3)) / 3)
print('Output element bytes:', batch_concat().nbytes)


Repeated growth best seconds per call: 1.711134488383929e-05
One concatenate best seconds per call: 3.08333740880092e-06
Preallocated best seconds per call: 3.69467306882143e-06
Output element bytes: 160000


#### Synthetic Time Series Example (Optional)

The following invented daily observations illustrate concatenation, not actual market prices or trading calendars. The same four columns remain in the same order in every input. Timing results above apply only to those inputs and do not measure peak memory.


In [ ]:
# Simulate daily stock prices for different months
np.random.seed(42)
jan_prices = np.random.uniform(100, 120, (31, 4))  # 31 days, 4 stocks
feb_prices = np.random.uniform(98, 118, (28, 4))   # 28 days, 4 stocks  
mar_prices = np.random.uniform(102, 122, (31, 4))  # 31 days, 4 stocks

stock_names = ['Asset A', 'Asset B', 'Asset C', 'Asset D']
print("Days per month:", jan_prices.shape[0], feb_prices.shape[0], mar_prices.shape[0])

# Combine quarterly data
q1_prices = np.concatenate((jan_prices, feb_prices, mar_prices), axis=0)
print("Q1 combined shape:", q1_prices.shape)

# Calculate monthly averages
monthly_avg = np.array([
    np.mean(jan_prices, axis=0),
    np.mean(feb_prices, axis=0),  
    np.mean(mar_prices, axis=0)
])

months = ['January', 'February', 'March']
for i, month in enumerate(months):
    print(f"{month:8}: " + " | ".join([f"{stock}: ${avg:.2f}" 
                                      for stock, avg in zip(stock_names, monthly_avg[i])]))

# Add new features (volume data) using horizontal concatenation
np.random.seed(42)
q1_volumes = np.random.randint(1000000, 5000000, (q1_prices.shape[0], 4))
q1_complete = np.hstack((q1_prices, q1_volumes))

print("\nWith volume data:", q1_complete.shape)
print("Sample day 1 prices:", q1_complete[0, :4])
print("Sample day 1 volumes:", q1_complete[0, 4:].astype(int))


📊 FINANCIAL DATA PROCESSING
Stock data shape - Days × Stocks:
January: (31, 4) (31 days)
February: (28, 4) (28 days)
March: (31, 4) (31 days)

Q1 Combined: (90, 4) (90 total days)

Monthly averages shape: (3, 4)
Monthly averages:
January : Asset A: $109.57 | Asset B: $109.65 | Asset C: $108.89 | Asset D: $110.21
February: Asset A: $106.62 | Asset B: $106.70 | Asset C: $109.63 | Asset D: $108.16
March   : Asset A: $113.36 | Asset B: $112.84 | Asset C: $111.29 | Asset D: $110.70

With volume data: (90, 8)
Columns: ['Asset A', 'Asset B', 'Asset C', 'Asset D'] (prices) + ['Asset A', 'Asset B', 'Asset C', 'Asset D'] (volumes)
Sample day 1: Prices=[107.49080238 119.01428613 114.63987884 111.97316968]
             Volumes=[3219110 3768307 3229084 4511566]


###  Concatenation Quick Reference

| **Operation** | **Function** | **Syntax** | **Result** |
|---------------|--------------|------------|------------|
| **Vertical Stack** | `vstack()` | `np.vstack((A, B))` | More rows for 2D inputs |
| **Horizontal Stack** | `hstack()` | `np.hstack((A, B))` | More columns for 2D inputs; longer vector for 1D |
| **General Concat** | `concatenate()` | `np.concatenate((A, B), axis=n)` | Custom axis |
| **3D Stack** | `dstack()` | `np.dstack((A, B))` | Depth dimension |
| **Column Combine** | `column_stack()` | `np.column_stack((v1, v2))` | Vectors → columns |

###  Common Pitfalls and Solutions

| **Problem** | **Cause** | **Solution** |
|-------------|-----------|--------------|
| `shapes not aligned` | Different dimensions | Use `reshape()` or `newaxis` |
| `axis out of bounds` | Wrong axis number | Check array dimensions with `.ndim` |
| `memory error` | Too many copies | Use pre-allocation or batch operations |
| `1D array issues` | Mixed 1D/2D arrays | Use `vstack`/`hstack` or reshape consistently |


## Practice Activity: Shapes, Sales, and Search {#practice-activity-shapes-sales-and-search}

**Goal:** Use array shapes to calculate and interpret a small sales report. **Time:** about 35–45 minutes. Use `activity05.ipynb` from the [practice kit](downloads/numpy-fundamentals-practice.zip). Submit `activity05.html` through the Chapter 5 Canvas quiz's final upload question.

**This section contains the complete activity instructions.** The starter supplies the inputs and work spaces. Optional benchmarks, the capital exercise, and other extensions are not required. These invented data describe units sold for one day, with stores in rows and products in columns:

```python
stores = np.array(['North', 'South', 'West'])
products = np.array(['Notebook', 'Pen', 'Folder', 'Marker'])
units = np.array([[12, 20, 8, 10], [10, 15, 12, 8], [12, 18, 9, 10]])
prices = np.array([5.0, 2.0, 3.0, 4.0])
store_factors = np.array([1.0, 0.9, 0.8])
new_store_units = np.array([9, 16, 10, 7])
```

Prices are dollars per unit; the factors are hypothetical multipliers applied to each store's entire revenue row. There are no missing values.

### A. Predict Shapes and Select Values

- Replace `Your Name` in the Raw title metadata and Markdown name field. Run the supplied imports and inputs.
- Display `units.shape`, `units.ndim`, `units.size`, and `units.dtype`; explain both axes.
- Before running them, predict the values and shapes of `units[:, 1]` and `units[:, 1:2]`. Run both and explain why their dimensions differ.
- Select the first two stores and the last two products with one two-dimensional slice. Display its values and shape.

### B. Broadcast by Product and by Store

- Calculate `revenue = units * prices`. Show the operand shapes, result shape, and result. Explain why each price matches a product and state the units.
- Explain why `revenue * store_factors` fails for these shapes. If you demonstrate the error, catch it with `try`/`except ValueError` so the notebook can run to completion.
- Reshape the factors into a column using `[:, None]` or `.reshape(-1, 1)`. Calculate and display `adjusted_revenue`, explaining why each store now receives its own multiplier. This is adjusted revenue, not profit.

### C. Summarize and Find Minimum/Maximum Records

- Use **unadjusted `revenue`** throughout this part. Calculate one revenue total per store and one per product with the appropriate axes. Display names beside totals and explain the shapes and dollar units.
- Find the store with the largest total using `argmax` and the store with the smallest total using `argmin`. Report each position, store name, and value.
- Find the largest individual store-product revenue with `np.max`. Use `argmax` and `unravel_index` to report the first matching store and product. Then use `np.argwhere(revenue == revenue.max())` to report all tied coordinates and their names. Explain the difference between a value and a position, and the tie rule.

### D. Edit Safely and Add a Store

- Start with `working = units.copy()`. Create `view = working[:, 0]` and `independent = working[:, 0].copy()` **before either edit**. Predict the effect of `view[0] = 0` and `independent[1] = 999`. Run them, then display `working`, `independent`, and original `units`. Explain which source changed and why.
- Using the original `units`, reshape `new_store_units` into one row and concatenate it on axis 0. Display the new shape and the new last row. Explain why shape `(4,)` cannot be directly concatenated with `(3, 4)` on axis 0, and why the product order must match.

### Render and Submit

Restart the kernel, run all cells in order, resolve unexpected errors, and save. Include your predictions, outputs, interpretations for A–D, and a short completion note. From the activity folder in the terminal, run:

```text
quarto render activity05.ipynb --to html
```

Follow the [Quarto refresher](vscode_setup.ipynb#render-and-submit-with-quarto): inspect the report and a copy opened outside the project folder. Confirm your name, code, outputs, and explanations are readable. Upload only `activity05.html` to the Chapter 5 Canvas quiz.

**HTML grading (16 points):** shapes and selections (3); broadcasting and units (4); axis summaries and min/max searches including ties (4); views, copies, and concatenation (4); readable named report and completion note (1).


## Extended Practice


### Capitals and Coordinate Distances {#capital-distances}

The supplied historical file is `data/country-capital-lat-long-population.csv`. Inspect its columns, missing coordinates, and country names. The reference country is `United States of America` in the `Country` column.

Use a simplified **Euclidean distance on latitude/longitude coordinates** for this first exercise. Its units are degrees in a coordinate plane, not kilometers; degrees of longitude do not represent the same ground distance everywhere, and the dateline creates a discontinuity. Treat this as practice with broadcasting and positional lookup, not a geographical distance ranking.

When you build the candidate table, remove the reference row first, and keep the candidate names and coordinate array in identical row order. Never use a fake large distance to exclude a record from a search: that would contaminate a subsequent maximum search.

**Tasks**

1. Load the data, drop rows with non-finite coordinates, and locate the single US reference row.
2. Compute the Euclidean distance from every other valid capital to the reference coordinates.
3. Find the closest capital, then the ten nearest and ten farthest capitals using a stable tie-breaking rule (`kind='stable'`).
4. Explain why an array position must be passed to `.iloc`, not `.loc`.


### Bonus: Nearest and Farthest Capitals on a Sphere

Use the haversine formula to estimate great-circle distances on a spherical Earth. Convert latitude/longitude to radians with `np.deg2rad`. If the two latitude/longitude pairs are `(φ₁, λ₁)` and `(φ₂, λ₂)`, compute

```text
h = sin²((φ₂ − φ₁)/2) + cos(φ₁) cos(φ₂) sin²((λ₂ − λ₁)/2)
distance = 2 × R × arcsin(sqrt(h))
```

Use `R = 6371.0` kilometers and `np.clip(h, 0, 1)` to handle floating-point roundoff. This is a spherical approximation, not an exact ellipsoidal geodesic or a travel route.

Reuse the valid candidates with the US reference excluded. Find the ten nearest and ten farthest, keep a stable input-order tie rule, and report names, coordinates, and kilometers. Compare the rankings with the coordinate-plane calculation and explain why they can differ. This bonus is not part of the Chapter 5 Canvas activity.


## Before You Move On {#before-you-move-on}

You should be able to predict the shape of an operation, identify the meaning of each axis, and distinguish a numerical result from its position. Check memory sharing before editing a slice and keep names aligned with the positions of your arrays.

Next, [Pandas Intermediate](data_types_in_pandas.ipynb) connects these array operations to labeled tables, alignment, and transformations. [NumPy Intermediate](vectorized_numpy.ipynb) later develops matrix computations and further vectorization.

References: [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html), [copies and views](https://numpy.org/doc/stable/user/basics.copies.html), [argmax](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html), and [argsort](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html).
